# CubeMind two-stage validation (Colab Blackwell)

Pre-flight check for the H200 production run. Validates the full pipeline at a tiny scale (~30M params, 2K steps) so we catch any bugs in the two-stage flow BEFORE spending H200 hours.

**Stages:**
1. **Stage 1** — LM pretrain on c4_realnewslike with the full hybrid stack (binding head + MoE + local attention + hippocampal memory + hypergrad)
2. **Stage 2** — load stage 1 checkpoint, freeze backbone, fine-tune the 5 multitask heads on Gemini+SVC rows

If both stages run cleanly here, the same script runs at 200M scale on H200 with `run_h200.sh`.

## 1. Setup

In [ ]:
import os, shutil, time
from pathlib import Path

# Clone cubemind (dev branch — has the two-stage flags)
%cd /content
if not Path('/content/cubemind').exists():
    !git clone --branch dev https://github.com/Grillcheese-AI/cubemind.git
%cd /content/cubemind
!git fetch origin && git checkout dev && git pull

# Mount Drive for the c4 corpus + tokenizer (assumes you've already
# uploaded grillcheese_spm32k_v2.model to Drive once)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.pop('MPLBACKEND', None)

In [ ]:
# Install minimum deps (PyTorch already on Colab images)
!pip install -q tokenizers sentencepiece python-dotenv pymupdf scikit-learn

In [ ]:
# Verify the data files are accessible. Adjust paths for your Drive layout.
TOKENIZER_PATH = '/content/drive/MyDrive/grillcheese_training_data/tokenizer/grillcheese_spm32k_v2.model'
LM_DATA_PATH   = '/content/drive/MyDrive/grillcheese_training_data/allenai_c4_realnewslike.500m_tokens.txt'
MT_DATA_PATH   = '/content/drive/MyDrive/grillcheese_training_data/multitask_combined.jsonl'

for p, label in [(TOKENIZER_PATH, 'tokenizer'),
                 (LM_DATA_PATH, 'LM corpus'),
                 (MT_DATA_PATH, 'multitask JSONL')]:
    if Path(p).exists():
        sz = Path(p).stat().st_size / 1e6
        print(f'  ✓  {label:18s} {sz:>8.1f} MB  {p}')
    else:
        print(f'  ✗  {label:18s} NOT FOUND  {p}')

## 2. Stage 1 — small LM pretrain (~10-15 min on Blackwell)

Tiny model: d=384, 6 layers, 768 d_ffn, ~30M params. 2000 steps so we get a real best.pt to feed stage 2.

In [ ]:
out = Path('/content/cubemind/sandbox/mingru_baseline/results_torch')
if out.exists():
    shutil.move(str(out), str(out.with_name(f'results_torch_pre_validate_{int(time.time())}')))
out.mkdir(parents=True, exist_ok=True)

!python -u sandbox/mingru_baseline/train_torch.py \
    --tokenizer-path "{TOKENIZER_PATH}" \
    --d-model 384 --n-layers 6 --d-ffn 768 --vocab 32128 --seq-len 256 \
    --steps 2000 --batch-size 16 --grad-accum 2 \
    --lr 6e-4 --min-lr 6e-5 --warmup 200 \
    --dtype bf16 --prompts sandbox/mingru_baseline/prompts_news.txt \
    --data-path "{LM_DATA_PATH}" \
    --vsa-binding-head --vsa-binding-d 10240 \
    --moe --moe-experts 4 --moe-top-k 2 \
    --attention --attn-heads 4 --attn-window 64 --attn-every-n 3 \
    --memory --mem-max 100 --mem-every-n 3 \
    --hypergrad \
    --log-every 50 --eval-every 500 --ckpt-every 500

In [ ]:
# Save stage 1 best so stage 2 can find it (move out of the results dir
# we're about to clobber).
import shutil
stage1_best = Path('/content/cubemind/sandbox/mingru_baseline/results_torch/best.pt')
stage1_saved = Path('/content/stage1_best.pt')
assert stage1_best.exists(), 'stage 1 produced no best.pt — check the train log above'
shutil.copy(str(stage1_best), str(stage1_saved))
print(f'  stage 1 best: {stage1_saved} ({stage1_saved.stat().st_size/1e6:.1f} MB)')

## 3. Stage 2 — multitask head fine-tune on frozen base (~5-10 min)

Loads stage 1 weights, freezes the backbone, trains only the 5 aux heads + the binding head's query projection. Should report per-head accuracies climbing across the run.

In [ ]:
# Archive stage 1 results so stage 2 has a clean output dir.
out = Path('/content/cubemind/sandbox/mingru_baseline/results_torch')
if out.exists():
    shutil.move(str(out), str(out.with_name(f'results_torch_stage1_{int(time.time())}')))
out.mkdir(parents=True, exist_ok=True)

!python -u sandbox/mingru_baseline/train_torch.py \
    --tokenizer-path "{TOKENIZER_PATH}" \
    --d-model 384 --n-layers 6 --d-ffn 768 --vocab 32128 --seq-len 256 \
    --steps 1000 --batch-size 32 --grad-accum 1 \
    --lr 2e-4 --min-lr 2e-5 --warmup 100 \
    --dtype bf16 \
    --use-jsonl-dataset --jsonl-path "{MT_DATA_PATH}" \
    --aux-opcode-loss-weight 0.4 \
    --aux-intent-loss-weight 0.2 \
    --aux-schema-loss-weight 0.2 \
    --vsa-binding-head --vsa-binding-d 10240 \
    --moe --moe-experts 4 --moe-top-k 2 \
    --attention --attn-heads 4 --attn-window 64 --attn-every-n 3 \
    --memory --mem-max 100 --mem-every-n 3 \
    --hypergrad \
    --init-from "{stage1_saved}" --freeze-backbone \
    --log-every 25 --eval-every 200 --ckpt-every 500

## 4. Validation

If both stages completed without errors AND stage 2's eval log shows head accuracies >0 (and ideally climbing), the two-stage flow works. Same script + same flags runs at 200M scale on H200.

In [ ]:
import json
summary_path = Path('/content/cubemind/sandbox/mingru_baseline/results_torch/summary.json')
if summary_path.exists():
    s = json.loads(summary_path.read_text())
    print(f'  params:        {s["params"]:,}')
    print(f'  final val CE:  {s["final_val_ce"]:.4f}')
    print(f'  best val CE:   {s["best_val_ce"]:.4f}')
    print(f'  steps:         {s["steps"]:,}')
    print(f'  tokens seen:   {s["tokens_seen"]:,}')
    print(f'  elapsed min:   {s["elapsed_min"]:.1f}')
    print(f'  tok/s:         {s["tokens_per_sec"]:,.0f}')
else:
    print('  no summary.json — stage 2 may have failed')